# Task 23 - Rating -> Sentiment Categories (dynamic thresholds, validation, encoding schemes)

**Goal:** encode ratings as **negative / neutral / positive** using thresholds derived from the *statistical distribution* of ratings (not fixed cut-offs), validate the labels with sentiment analysis, and test several encoding schemes to find the one with the best predictive power.

**Data mapping:** `rating` is a **product-level average** (1-5) and is skewed high, so fixed star cut-offs would put almost every product in "positive". Ratings from products with few votes are noisy, so the basis is the **shrunk rating**: `(rating*n + global_mean*m) / (n + m)` with `m = PRIOR_STRENGTH`. A second basis uses the **vote mix** from `rating_*_count` (share of 4-5 stars minus share of 1-2 stars) when those columns exist.

**Threshold schemes (all recomputed from the data on every run):** fixed classic (3 / 4), tertiles, outer quartiles, mean +- 0.5 sd, median +- scaled MAD, 1-D K-means, Gaussian mixture, vote-share index.

**Validation with sentiment analysis (title text, the only text available):**
1. VADER compound by class should rise from negative to positive (monotonic check, Kruskal test, eta-squared).
2. A TF-IDF logistic regression should separate positive from negative titles better than chance (CV AUC).
3. Vote mix by class should be ordered as well.

**Encoding schemes tested:** ordinal (0/1/2), one-hot, positive-vs-rest, negative-vs-rest, fold-wise target encoding, plus the raw rating and "no sentiment" baselines. **Predictive power** = CV AUC for predicting above-median `units_sold` from product attributes plus the encoded sentiment (encoding fitted inside each fold).

**Input:** newest file before this task. **Outputs:** `data/processed/womens_clothing_rating_sentiment_encoded.csv`, `reports/task_23_threshold_schemes.csv`, `reports/task_23_encoding_comparison.csv`, `reports/task_23_sentiment_validation.csv`, `reports/task_23_analysis_status.csv`

In [ ]:
import re, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

def find_project_root():
    """Walk up from the notebook folder until a folder containing data/processed is found."""
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "data" / "processed").exists():
            return cand
    return start.parent

ROOT = find_project_root()
PROC, REPORTS = ROOT / "data" / "processed", ROOT / "reports"
RAW = ROOT / "data" / "raw" / "womens_clothing.csv"
PROC.mkdir(parents=True, exist_ok=True); REPORTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def pick(df, *candidates, required=True):
    """Return the real column name matching any candidate (ignores case, spaces, underscores)."""
    lookup = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if _norm(cand) in lookup:
            return lookup[_norm(cand)]
    if required:
        raise KeyError(f"None of {candidates} found. Columns: {list(df.columns)}")
    return None

def load_first(*paths):
    """Load the first existing CSV, so each task builds on the newest previous output."""
    for p in paths:
        if Path(p).exists():
            d = pd.read_csv(p)
            d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
            print(f"Loaded: {Path(p).name}  shape={d.shape}")
            return d, Path(p)
    raise FileNotFoundError("None of these files exist: " + ", ".join(str(p) for p in paths))

def add_from_raw(df, *names):
    """Borrow columns missing from df (e.g. title, tags) from data/raw, matched on product_id."""
    missing = [n for n in names if pick(df, n, required=False) is None]
    if not missing:
        return df
    raw = pd.read_csv(RAW); raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")]
    kd, kr = pick(df, "product_id", required=False), pick(raw, "product_id", required=False)
    if kd is None or kr is None:
        raise KeyError(f"Need product_id in both files to borrow {missing} from raw")
    got = [pick(raw, n) for n in missing if pick(raw, n, required=False)]
    if not got:
        raise KeyError(f"{missing} not found in raw file either")
    add = raw.drop_duplicates(kr)[[kr] + got].rename(columns={kr: kd})
    out = df.merge(add, on=kd, how="left")
    print("Borrowed from raw:", got)
    return out

# Newest cumulative dataset first; each task reads the newest file produced BEFORE it.
CHAIN = ["womens_clothing_high_cardinality_encoded.csv", "womens_clothing_engagement_features.csv",
         "womens_clothing_sentiment_features.csv", "womens_clothing_age_segmented.csv",
         "womens_clothing_review_text_features.csv", "womens_clothing_market_basket_features.csv",
         "womens_clothing_product_text_features.csv", "womens_clothing_product_tag_features.csv",
         "womens_clothing_product_segmented.csv", "womens_clothing_final.csv"]

# Tasks 19-21 outputs sit in front of the older chain.
LATE = ["womens_clothing_lifecycle_features.csv", "womens_clothing_tfidf_features.csv", "womens_clothing_channel_interaction_features.csv"]
FULL = LATE + CHAIN

# Tasks 22-24 outputs sit in front of everything.
LATE2 = ["womens_clothing_rating_sentiment_encoded.csv", "womens_clothing_pca_features.csv"]
ALL = LATE2 + FULL

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
PRIOR_STRENGTH = 20
CLASS_NAMES = ["negative", "neutral", "positive"]

## 1. Load data and build the rating basis

In [ ]:
df, src = load_first(*[PROC / n for n in ALL[1:]])
df = add_from_raw(df, "title")
try: df = add_from_raw(df, "rating_one_count", "rating_two_count", "rating_three_count", "rating_four_count", "rating_five_count")
except KeyError as e: print("Vote-count columns unavailable ->", str(e)[:90])
rating_col = pick(df, "rating"); rc_col = pick(df, "rating_count"); units_col = pick(df, "units_sold"); title_col = pick(df, "title")
cnt_cols = {k: pick(df, f"rating_{k}_count", required=False) for k in ["one", "two", "three", "four", "five"]}
gm = np.average(df[rating_col], weights=df[rc_col].clip(lower=1))
df["rs_rating_shrunk"] = (df[rating_col] * df[rc_col] + gm * PRIOR_STRENGTH) / (df[rc_col] + PRIOR_STRENGTH)
basis = df["rs_rating_shrunk"]
has_votes = all(cnt_cols.values())
if has_votes:
    tot = sum(df[c] for c in cnt_cols.values()).replace(0, np.nan)
    df["rs_vote_net_index"] = ((df[cnt_cols["four"]] + df[cnt_cols["five"]] - df[cnt_cols["one"]] - df[cnt_cols["two"]]) / tot).fillna(0)
    df["rs_vote_pos_share"] = ((df[cnt_cols["four"]] + df[cnt_cols["five"]]) / tot).fillna(0)
print(f"Products: {len(df):,} | global mean rating {gm:.3f} | shrunk rating: min {basis.min():.2f}, median {basis.median():.2f}, max {basis.max():.2f} | vote counts available: {has_votes}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[rating_col], bins=40, ax=ax[0], color="#c44e52"); ax[0].set_title("Raw product rating")
sns.histplot(basis, bins=40, ax=ax[1], color="#4c72b0"); ax[1].set_title("Shrunk rating (used for thresholds)"); plt.tight_layout(); plt.show()

## 2. Dynamic threshold schemes

In [ ]:
def classify(x, lo, hi): return np.where(x < lo, 0, np.where(x >= hi, 2, 1))
x = basis.values; schemes = {}
schemes["fixed_classic (3 / 4)"] = (3.0, 4.0, "rating")
q1, q2 = np.quantile(x, [1 / 3, 2 / 3]); schemes["tertiles"] = (q1, q2, "rating")
q1, q2 = np.quantile(x, [.25, .75]); schemes["outer_quartiles"] = (q1, q2, "rating")
m, s = x.mean(), x.std(); schemes["mean_pm_0.5sd"] = (m - .5 * s, m + .5 * s, "rating")
med = np.median(x); mad = 1.4826 * np.median(np.abs(x - med)); schemes["median_pm_0.75mad"] = (med - .75 * mad, med + .75 * mad, "rating")
c = np.sort(KMeans(3, n_init=10, random_state=RANDOM_STATE).fit(x.reshape(-1, 1)).cluster_centers_.ravel()); schemes["kmeans_1d"] = ((c[0] + c[1]) / 2, (c[1] + c[2]) / 2, "rating")
gmm = GaussianMixture(3, random_state=RANDOM_STATE, n_init=3).fit(x.reshape(-1, 1)); grid = np.linspace(x.min(), x.max(), 2000)
order = np.argsort(gmm.means_.ravel()); post = gmm.predict_proba(grid.reshape(-1, 1))[:, order]; lab = post.argmax(axis=1)
schemes["gmm_3"] = (grid[np.argmax(lab >= 1)] if (lab >= 1).any() else q1, grid[np.argmax(lab >= 2)] if (lab >= 2).any() else q2, "rating")
if has_votes:
    v = df["rs_vote_net_index"].values; mv, sv = v.mean(), v.std(); schemes["vote_net_mean_pm_0.5sd"] = (mv - .5 * sv, mv + .5 * sv, "vote")
labels, rows = {}, []
for name, (lo, hi, kind) in schemes.items():
    arr = classify(df["rs_vote_net_index"].values if kind == "vote" else x, lo, hi); labels[name] = arr
    sh = np.bincount(arr, minlength=3) / len(arr)
    rows.append({"scheme": name, "basis": "vote net index" if kind == "vote" else "shrunk rating", "neg_below": lo, "pos_from": hi,
                 "share_negative": sh[0], "share_neutral": sh[1], "share_positive": sh[2], "min_class_share": sh.min()})
thr = pd.DataFrame(rows); thr.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5)); sns.histplot(basis, bins=60, color="#dddddd", ax=ax)
for i, (name, (lo, hi, kind)) in enumerate(schemes.items()):
    if kind == "rating": ax.axvline(lo, lw=1, color=f"C{i}", label=name); ax.axvline(hi, lw=1, color=f"C{i}")
ax.legend(fontsize=7, ncol=2); ax.set_title("Threshold positions on the rating distribution"); plt.show()

## 3. Validate the labels with sentiment analysis

In [ ]:
analyzer = SentimentIntensityAnalyzer()
text = df[title_col].astype("string").fillna("").str.strip()
has_text = (text.str.len() > 0).values
vader = np.array([analyzer.polarity_scores(t)["compound"] if t else 0.0 for t in text])
rows = []
cvt = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
for name, lab_ in labels.items():
    g = [vader[(lab_ == k) & has_text] for k in range(3)]
    means = [gi.mean() if len(gi) else np.nan for gi in g]
    H, p = stats.kruskal(*[gi for gi in g if len(gi) > 1]) if sum(len(gi) > 1 for gi in g) >= 2 else (np.nan, np.nan)
    eta = (H - 2) / (has_text.sum() - 3) if not np.isnan(H) else np.nan
    rho = stats.spearmanr(lab_[has_text], vader[has_text])[0]
    sel = has_text & (lab_ != 1)                                              # positive vs negative titles
    if sel.sum() > 100 and len(set(lab_[sel])) == 2:
        pr = cross_val_predict(make_pipeline(TfidfVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2)), LogisticRegression(max_iter=1000, C=0.5)),
                               text[sel].astype(str), (lab_[sel] == 2).astype(int), cv=cvt, method="predict_proba")[:, 1]
        text_auc = roc_auc_score((lab_[sel] == 2).astype(int), pr)
    else: text_auc = np.nan
    vote_means = [df["rs_vote_pos_share"].values[lab_ == k].mean() for k in range(3)] if has_votes else [np.nan] * 3
    rows.append({"scheme": name, "vader_neg": means[0], "vader_neu": means[1], "vader_pos": means[2], "vader_monotonic": bool(means[0] <= means[1] <= means[2]),
                 "kruskal_p": p, "eta_squared": eta, "spearman_label_vs_vader": rho, "title_tfidf_auc_pos_vs_neg": text_auc,
                 "vote_pos_share_neg": vote_means[0], "vote_pos_share_neu": vote_means[1], "vote_pos_share_pos": vote_means[2],
                 "vote_monotonic": bool(vote_means[0] <= vote_means[1] <= vote_means[2]) if has_votes else np.nan})
val = pd.DataFrame(rows); val.round(4)

## 4. Encoding schemes and predictive power

Downstream target: **above-median `units_sold`**. Features: price, retail price, discount, shipping price, badges, ad boosts, merchant rating and size, title features, category and **log rating count as a control** (shrinkage ties the rating basis to `rating_count`, which tracks sales, so the sentiment feature must be judged by its gain *beyond* popularity). The encoding is fitted **inside each CV fold** (matters for target encoding).

In [ ]:
disc = None
feat = pd.DataFrame(index=df.index)
for name, cand in {"price": ["price"], "retail_price": ["retail_price"], "shipping_price": ["shipping_option_price"], "badges": ["badges_count"], "ads": ["uses_ad_boosts"],
                   "merchant_rating": ["merchant_rating"], "merchant_rating_count": ["merchant_rating_count"], "express": ["shipping_is_express"],
                   "urgency": ["has_urgency_banner"], "inventory": ["inventory_total"], "ttl_words": ["ttl_word_count"], "ttl_sent": ["ttl_sent_compound"], "n_tags": ["mb_n_tags"]}.items():
    c = pick(df, *cand, required=False)
    if c is not None: feat[name] = pd.to_numeric(df[c], errors="coerce")
feat["log_rating_count"] = np.log1p(df[rc_col])        # control: shrinkage ties the rating basis to rating_count, which tracks sales
if "price" in feat and "retail_price" in feat: feat["discount_pct"] = (1 - feat["price"] / feat["retail_price"].replace(0, np.nan)).clip(-1, 1)
cat = pick(df, "mb_category", required=False)
if cat: feat = pd.concat([feat, pd.get_dummies(df[cat].astype("string"), prefix="cat", dtype=int)], axis=1)
feat = feat.fillna(feat.median(numeric_only=True)).fillna(0)
y_sales = (df[units_col] > df[units_col].median()).astype(int).values
print("Base features:", list(feat.columns))

def encode_fold(lab_tr, lab_te, scheme_kind, ytr):
    """Return (train_cols, test_cols) for an encoding fitted on the training fold only."""
    if scheme_kind == "ordinal":      return lab_tr[:, None], lab_te[:, None]
    if scheme_kind == "one_hot":      return np.eye(3)[lab_tr], np.eye(3)[lab_te]
    if scheme_kind == "pos_vs_rest":  return (lab_tr == 2)[:, None].astype(float), (lab_te == 2)[:, None].astype(float)
    if scheme_kind == "neg_vs_rest":  return (lab_tr == 0)[:, None].astype(float), (lab_te == 0)[:, None].astype(float)
    if scheme_kind == "target_enc":
        g = ytr.mean(); mp = {k: ((ytr[lab_tr == k].sum() + 20 * g) / ((lab_tr == k).sum() + 20)) for k in range(3)}
        return np.vectorize(mp.get)(lab_tr)[:, None].astype(float), np.vectorize(mp.get)(lab_te)[:, None].astype(float)

def cv_auc(lab_, kind):
    aucs = []
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE).split(feat, y_sales):
        Xtr, Xte = feat.values[tr], feat.values[te]
        if kind != "none":
            a, b = (basis.values[tr][:, None], basis.values[te][:, None]) if kind == "raw_rating" else encode_fold(lab_[tr], lab_[te], kind, y_sales[tr])
            Xtr, Xte = np.hstack([Xtr, a]), np.hstack([Xte, b])
        rf = RandomForestClassifier(150, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, y_sales[tr])
        aucs.append(roc_auc_score(y_sales[te], rf.predict_proba(Xte)[:, 1]))
    return np.mean(aucs), np.std(aucs) / np.sqrt(len(aucs))

rows = [{"scheme": "(none)", "encoding": "no sentiment feature", "n_cols": 0, **dict(zip(["cv_auc", "cv_se"], cv_auc(None, "none")))},
        {"scheme": "(raw shrunk rating)", "encoding": "continuous rating, no encoding", "n_cols": 1, **dict(zip(["cv_auc", "cv_se"], cv_auc(None, "raw_rating")))}]
for name, lab_ in labels.items():
    for kind, ncol in [("ordinal", 1), ("one_hot", 3), ("pos_vs_rest", 1), ("neg_vs_rest", 1), ("target_enc", 1)]:
        a, se = cv_auc(lab_, kind); rows.append({"scheme": name, "encoding": kind, "n_cols": ncol, "cv_auc": a, "cv_se": se})
enc = pd.DataFrame(rows).sort_values("cv_auc", ascending=False).reset_index(drop=True)
base_auc = enc.loc[enc["scheme"] == "(none)", "cv_auc"].iloc[0]; enc["gain_vs_none"] = enc["cv_auc"] - base_auc
enc.round(4).head(15)

In [ ]:
top = enc[~enc["scheme"].str.startswith("(")].copy(); top["cfg"] = top["scheme"].str[:22] + " | " + top["encoding"]
plt.figure(figsize=(9, 7)); sns.barplot(data=top.head(20), x="cv_auc", y="cfg", color="#4c72b0"); plt.axvline(base_auc, color="k", ls="--", label="no sentiment feature"); plt.legend()
plt.xlim(top["cv_auc"].min() - .01, top["cv_auc"].max() + .01); plt.title("Downstream CV AUC (predict high sales)"); plt.tight_layout(); plt.show()

## 5. Choose the scheme

Candidates within 1 SE of the best AUC are kept; among them the scheme with the strongest **text validation** (TF-IDF AUC pos-vs-neg, then VADER monotonicity) wins, preferring the simplest encoding (fewest columns). If no scheme beats the no-sentiment baseline, the notebook still picks the best-validated, simplest scheme and says so.

In [ ]:
ok = enc[~enc["scheme"].str.startswith("(")]
best = ok.iloc[0]; cand = ok[ok["cv_auc"] >= best["cv_auc"] - best["cv_se"]].merge(val[["scheme", "title_tfidf_auc_pos_vs_neg", "vader_monotonic", "vote_monotonic"]], on="scheme", how="left")
cand["vm"] = cand["vader_monotonic"].fillna(False).astype(int)
choice = cand.sort_values(["title_tfidf_auc_pos_vs_neg", "vm", "n_cols"], ascending=[False, False, True]).iloc[0]
beats = bool(choice["cv_auc"] > base_auc)
print("Chosen scheme:", choice["scheme"], "| encoding:", choice["encoding"], "| CV AUC", round(choice["cv_auc"], 4), "vs none", round(base_auc, 4), "| beats baseline:", beats)
sc_name = choice["scheme"]; lab_final = labels[sc_name]; lo, hi, kind_ = schemes[sc_name]
print(f"Thresholds ({'vote net index' if kind_ == 'vote' else 'shrunk rating'}): negative < {lo:.3f} <= neutral < {hi:.3f} <= positive")

## 6. Create the final features

In [ ]:
out = df.copy()
out["rating_sentiment"] = pd.Series(lab_final, index=df.index).map(dict(enumerate(CLASS_NAMES)))
out["rating_sentiment_ord"] = lab_final
for i, n in enumerate(CLASS_NAMES): out[f"rating_sentiment_{n}"] = (lab_final == i).astype(int)
out["rating_sentiment_pos_vs_rest"] = (lab_final == 2).astype(int)
# out-of-fold target encoding (5-fold) so a product's own sales never feed its encoding
te = np.zeros(len(df))
for tr, te_i in StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE).split(feat, y_sales):
    a, b = encode_fold(lab_final[tr], lab_final[te_i], "target_enc", y_sales[tr]); te[te_i] = b.ravel()
out["rating_sentiment_target_enc_oof"] = te
out["rating_sentiment_scheme"] = sc_name
out["rating_sentiment_neg_threshold"] = lo; out["rating_sentiment_pos_threshold"] = hi
out["rating_confident"] = (df[rc_col] >= 20).astype(int)
print(out["rating_sentiment"].value_counts(normalize=True).round(3))
pd.crosstab(out["rating_sentiment"], out["rating_confident"], normalize="columns").round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
sns.boxplot(data=out, x="rating_sentiment", y="rs_rating_shrunk", order=CLASS_NAMES, ax=ax[0]); ax[0].set_title("Shrunk rating by class")
sns.boxplot(x=out["rating_sentiment"], y=vader, order=CLASS_NAMES, ax=ax[1], showfliers=False); ax[1].set_title("VADER (title) by class")
sns.barplot(data=out, x="rating_sentiment", y=(df[units_col] > df[units_col].median()).astype(int), order=CLASS_NAMES, ax=ax[2], color="#55a868"); ax[2].set_title("Share of high-sales products by class")
plt.tight_layout(); plt.show()

## 7. Save outputs

In [ ]:
out_path = PROC / "womens_clothing_rating_sentiment_encoded.csv"; out.to_csv(out_path, index=False)
thr.assign(selected=(thr["scheme"] == sc_name)).round(5).to_csv(REPORTS / "task_23_threshold_schemes.csv", index=False)
enc.round(5).to_csv(REPORTS / "task_23_encoding_comparison.csv", index=False)
val.assign(selected=(val["scheme"] == sc_name)).round(5).to_csv(REPORTS / "task_23_sentiment_validation.csv", index=False)
base_cols = pd.read_csv(src, nrows=1).columns
status = pd.DataFrame([{"task": "Task 23 - Rating sentiment encoding", "status": "completed", "run_at": datetime.now().isoformat(timespec="seconds"),
    "input_file": src.name, "rows": len(out), "chosen_scheme": sc_name, "chosen_encoding": choice["encoding"], "neg_threshold": round(lo, 4), "pos_threshold": round(hi, 4),
    "cv_auc_chosen": round(choice["cv_auc"], 4), "cv_auc_no_sentiment": round(base_auc, 4), "beats_baseline": beats,
    "vader_monotonic": bool(val.loc[val["scheme"] == sc_name, "vader_monotonic"].iloc[0]), "title_tfidf_auc": round(float(val.loc[val["scheme"] == sc_name, "title_tfidf_auc_pos_vs_neg"].iloc[0]), 4),
    "columns_added": len([c for c in out.columns if c not in base_cols]), "output_dataset": out_path.name}])
status.to_csv(REPORTS / "task_23_analysis_status.csv", index=False); status.T

## 8. Key takeaways (fill in after running)

- Where did the dynamic thresholds land compared with the fixed 3 / 4 star cut-offs, and how balanced are the classes?
- Do VADER and the title TF-IDF model agree with the labels (monotonic VADER, AUC above 0.5)? Titles are marketing copy, so weak validation is expected; vote-share ordering is the stronger check.
- Did any encoding beat "no sentiment feature" on sales prediction? A gain near zero means rating sentiment adds little beyond price and merchant attributes.
- `rating_sentiment*` columns are derived from the rating, so avoid them when the target is the rating itself.